In [1]:

# =========================================================
# Padronizador de NFs (HTB + DANFE + NFS-e)
# Versão limpa (sem JSON) - pronto para Jupyter/py
# =========================================================

import os
import re
import shutil
import sys
import subprocess

# --------------------- Dependências ----------------------
def instalar(pacote):
    try:
        __import__(pacote)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pacote])

instalar("pdfplumber")
instalar("pandas")
instalar("tabulate")

import pdfplumber
import pandas as pd
from tabulate import tabulate

# =========================================================
# NF – EXTRAÇÃO PELO PDF
# =========================================================
def extrair_nf_pdf(texto: str):
    """
    Tenta extrair o número da NF do texto do PDF:
      1) NFS-e: "Número da Nota 000123"
      2) DANFE: pela chave de 44 dígitos (posições 26–34)
      3) Se constar "RPS", ignora (não é NF)
    """
    if not texto:
        return None

    texto = texto.replace("\n", " ")

    # 1) NFS-e -> "Número da Nota 000123"
    padrao_nfse = r"Número\s+da\s+Nota\s+0*(\d{4,9})"
    m = re.search(padrao_nfse, texto, re.IGNORECASE)
    if m:
        return str(int(m.group(1)))

    # 2) DANFE – chave 44 dígitos
    chaves = re.findall(r"\b\d{44}\b", texto.replace(" ", ""))
    if chaves:
        # NF fica nas posições 26–34 (índices 25..33); usamos 25:34
        return str(int(chaves[0][25:34]))

    # 3) Bloqueia RPS explicitamente
    if re.search(r"\bRPS\b", texto, re.IGNORECASE):
        return None

    return None

# =========================================================
# NF – FALLBACK PELO NOME DO ARQUIVO (HTB)
# =========================================================
def extrair_nf_nome_arquivo(nome: str):
    """
    Ex: "NFS 15401 - CGR09 - QUARENTA_PB.pdf" -> 15401
    """
    m = re.match(r"NFS\s+0*(\d+)", nome, re.IGNORECASE)
    if m:
        return str(int(m.group(1)))
    return None

# =========================================================
# VALOR TOTAL
# =========================================================
def extrair_valor_total(texto: str):
    """
    Varre padrões típicos de "VALOR TOTAL" em NFs/NFS-e.
    Retorna float (ponto como separador decimal) ou None.
    """
    if not texto:
        return None
    padrao = (
        r"(?:VALOR\s+TOTAL\s+DO\s+SERVIÇ?O|"
        r"VALOR\s+TOTAL\s+DA\s+NOTA|"
        r"TOTAL\s+DA\s+NOTA|"
        r"VALOR\s+TOTAL)"
        r"\s*(?:=|:)?\s*R?\$?\s*([\d\.,]+)"
    )
    m = re.search(padrao, texto, re.IGNORECASE)
    if m:
        try:
            return float(m.group(1).replace(".", "").replace(",", "."))
        except ValueError:
            return None
    return None

# =========================================================
# UF
# =========================================================
def extrair_uf(texto: str):
    if not texto:
        return None
    ufs = r"\b(AC|AL|AM|AP|BA|CE|DF|ES|GO|MA|MG|MS|MT|PA|PB|PE|PI|PR|RJ|RN|RO|RR|RS|SC|SE|SP|TO)\b"
    m = re.search(ufs, texto)
    return m.group(1) if m else None

# =========================================================
# LEITURA DO PDF
# =========================================================
def extrair_dados_pdf(caminho: str):
    """
    Extrai texto de todas as páginas e tenta obter:
      - nf (PDF, com fallback de nome no fluxo principal)
      - loja (via CNPJ do tomador/destinatário)
      - valor total
      - uf
    Regras loja:
      - CNPJ Pague Menos (06.xxxx/####-##): usa últimos 4 dígitos (sem zeros à esquerda)
      - CNPJ Imifarma (04.xxxx/####-##): força 7xxx (ajuste histórico)
    """
    texto = ""
    with pdfplumber.open(caminho) as pdf:
        for p in pdf.pages:
            texto += p.extract_text() or ""

    nf = extrair_nf_pdf(texto)

    # -------- CNPJ / LOJA --------
    loja = None
    cnpj_raw = None

    padroes_cnpj = [
        r"(06\.626\.253/\d{4}-\d{2})",   # Pague Menos
        r"(03\.641\.059/\d{4}-\d{2})",   # Extrafarma
        r"(78\.331\.899/\d{4}-\d{2})",   # Outro conhecido
        r"CNPJ\s*:?\s*(\d{2}\.\d{3}\.\d{3}/\d{4}-\d{2})"
    ]

    for ptt in padroes_cnpj:
        m = re.search(ptt, texto)
        if m:
            cnpj_raw = m.group(1)
            break

    if cnpj_raw:
        m_loja = re.search(r"/(\d{4})-", cnpj_raw)
        if m_loja:
            loja = m_loja.group(1)

            # IMIFARMA (04....)
            if cnpj_raw.startswith("04"):
                loja = "7" + loja[1:] if len(loja) >= 2 else loja

            # PAGUE MENOS (06....)
            if cnpj_raw.startswith("06"):
                loja = loja.lstrip("0") or "0"

    valor = extrair_valor_total(texto)
    uf = extrair_uf(texto)

    return nf, loja, valor, uf

# =========================================================
# PROCESSAMENTO DA PASTA
# =========================================================
def padronizar_pasta(pasta: str):
    destino = os.path.join(pasta, "NFs_Padronizadas")
    os.makedirs(destino, exist_ok=True)

    resumo = []

    for arq in os.listdir(pasta):
        if not arq.lower().endswith(".pdf"):
            continue

        caminho = os.path.join(pasta, arq)
        print(f"\n📂 Lendo: {arq}")

        nf, loja, valor, uf = extrair_dados_pdf(caminho)

        # ✅ FALLBACK PARA HTB (pelo nome do arquivo)
        if not nf:
            nf = extrair_nf_nome_arquivo(arq)

        if nf and loja:
            novo_nome = f"NF {nf} - LJ {loja}.pdf"
            status = "Renomeado"
        elif nf:
            novo_nome = f"NF {nf}.pdf"
            status = "Sem loja"
        else:
            novo_nome = arq
            status = "NF não encontrada"

        # Copia com o novo nome (se já existir, sobrescreve — ajuste se quiser safe copy)
        shutil.copy(caminho, os.path.join(destino, novo_nome))
        print("✅", novo_nome)

        valor_fmt = (
            f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
            if valor is not None else "-"
        )

        resumo.append([arq, nf or "-", loja or "-", uf or "-", valor_fmt, status])

    df = pd.DataFrame(
        resumo,
        columns=["Arquivo Original", "NF", "Loja", "UF", "Valor Total", "Status"]
    )

    print("\n📊 RESUMO FINAL\n")
    print(tabulate(df, headers="keys", tablefmt="fancy_grid"))

    df.to_excel(os.path.join(destino, "Resumo_NFs_Padronizadas.xlsx"), index=False)

# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    pasta = input("Digite o caminho da pasta com os PDFs: ").strip('"')
    if os.path.exists(pasta):
        padronizar_pasta(pasta)
    else:
        print("❌ Caminho inválido")


Digite o caminho da pasta com os PDFs:  C:\Users\126815\OneDrive - paguemenos.com.br\Área de Trabalho\NFS\2025\LOJAS NOVAS\HTB\22.01.2026



📂 Lendo: NFS 15589 - CD - CONDEPB.pdf
✅ NF 15589.pdf

📂 Lendo: NFS 15590 - MAN19.pdf
✅ NF 15590.pdf

📂 Lendo: NFS 15591 - NAV01.pdf
✅ NF 15591.pdf

📂 Lendo: NFS 15592 - IRA01.pdf
✅ NF 15592.pdf

📂 Lendo: NFS 15593 - IGA02.pdf
✅ NF 15593.pdf

📂 Lendo: NFS 15594 - HOR02.pdf
✅ NF 15594.pdf

📂 Lendo: NFS 15595 - 7187 - FOR164 - EF.pdf
✅ NF 15595.pdf

📊 RESUMO FINAL

╒════╤════════════════════════════════════╤═══════╤════════╤══════╤═══════════════╤══════════╕
│    │ Arquivo Original                   │    NF │ Loja   │ UF   │ Valor Total   │ Status   │
╞════╪════════════════════════════════════╪═══════╪════════╪══════╪═══════════════╪══════════╡
│  0 │ NFS 15589 - CD - CONDEPB.pdf       │ 15589 │ -      │ -    │ -             │ Sem loja │
├────┼────────────────────────────────────┼───────┼────────┼──────┼───────────────┼──────────┤
│  1 │ NFS 15590 - MAN19.pdf              │ 15590 │ -      │ -    │ -             │ Sem loja │
├────┼────────────────────────────────────┼───────┼────────┼────